In [1]:
source ../modules/setup.tcl

set year 2015
set day 7

aoc::get-puzzle $year $day 1
aoc::get-puzzle $year $day 2
set input [string trim [aoc::get-input $year $day]]
jupyter::display "text/markdown" "## Input\n```\n[string range $input 0 20]...\n```";

(cached)

--- Day 7: Some Assembly Required --- This year, Santa brought little Bobby Tables a set of wires and bitwise logic gates ! Unfortunately, little Bobby is a little under the recommended age range, and he needs help assembling the circuit . 
 Each wire has an identifier (some lowercase letters) and can carry a 16-bit signal (a number from 0 to 65535 ). A signal is provided to each wire by a gate, another wire, or some specific value. Each wire can only get a signal from one source, but can provide its signal to multiple destinations. A gate provides no signal until all of its inputs have a signal. 
 The included instructions booklet describes how to connect the parts together: x AND y -> z means to connect wires x and y to an AND gate, and then connect its output to wire z . 
 For example: 
 
 
 123 -> x means that the signal 123 is provided to wire x . 
 
 x AND y -> z means that the bitwise AND of wire x and wire y is provided to wire z . 
 
 p LSHIFT 2 -> q means that the value from wire p is left-shifted by 2 and then provided to wire q . 
 
 NOT e -> f means that the bitwise complement of the value from wire e is provided to wire f . 
 
 Other possible gates include OR ( bitwise OR ) and RSHIFT ( right-shift ). If, for some reason, you'd like to emulate the circuit instead, almost all programming languages (for example, C , JavaScript , or Python ) provide operators for these gates. 
 For example, here is a simple circuit: 
 123 -> x
456 -> y
x AND y -> d
x OR y -> e
x LSHIFT 2 -> f
y RSHIFT 2 -> g
NOT x -> h
NOT y -> i
 
 After it is run, these are the signals on the wires: 
 d: 72
e: 507
f: 492
g: 114
h: 65412
i: 65079
x: 123
y: 456
 
 In little Bobby's kit's instructions booklet (provided as your puzzle input), what signal is ultimately provided to wire a 
 ?

--- Part Two --- Now, take the signal you got on wire a , override wire b to that signal, and reset the other wires (including wire a ). What new signal is ultimately provided to wire a ?

(cached)

## Input
```
lf AND lq -> ls
iu RS...
```

In [2]:
proc parse {line} {
    regexp  {(.*) -> (.*)} $line -> cmd target
    set cmd [split $cmd]
    return [list $target $cmd]
}

proc longer {x y} {
    set ldiv [expr {[llength $x] - [llength $y]}]

        return $ldiv
  
}

proc simulate {input start} {
set input [lsort -command longer  [split $input \n]  ]
set queue $input
array set wires $start
while {[llength $queue] > 0} {
    set line [lindex $queue 0]
    lassign [parse $line] target cmd
    if {[info exists wires($target)]} {
        # puts "not overwriting $target: $cmd"
        set queue [lrange $queue 1 end]
        continue
    }
    if {[catch {



    switch -exact [llength $cmd] {
        1 {

            if {[string is integer $cmd]} {
                set wires($target) $cmd
            } else {
                set wires($target) $wires($cmd)
            }
                       
            }
        2 {
 
            lassign $cmd opcode src
            if {$opcode ne "NOT"} {
                error "unsupported opcode: $opcode"
            }
            if {[string is integer $src]} {
                set srcVal $src
            } else {
                set srcVal $wires($src)
            }
            set wires($target) [expr {~($srcVal)  & 0xFFFF}]

        
        }
        3 {
     
            lassign $cmd src1 opcode src2
            if {[string is integer $src1]} {
                set src1Val $src1
            } else {
                set src1Val $wires($src1)
            }
            if {[string is integer $src2]} {
                set src2Val $src2
            } else {
                set src2Val $wires($src2)
            }
            switch -exact $opcode {
                AND {
                    set wires($target) [expr {$src1Val & $src2Val & 0xFFFF}]
                }
                OR {
                    set wires($target) [expr {$src1Val | $src2Val  & 0xFFFF}]
                }
                LSHIFT {
                    set wires($target) [expr {$src1Val << $src2Val  & 0xFFFF}]
                }
                RSHIFT {
                    set wires($target) [expr {$src1Val >> $src2Val  & 0xFFFF}]
                }
                default {
                    error "unsupported opcode: $opcode"
                }
            }
        }
        default {
     
            error "unsupported number of arguments in: $cmd"
        
      } 
    }
      set queue [lrange $queue 1 end]
    } res]} {
    # puts $res
        set queue [lassign $queue start]
        lappend queue $start
        #puts [llength $queue]
    }
        
   }
   return [array get wires]

}


In [4]:
aoc::solve $input {
    set result2 0
    set result1 [dict get [simulate $input {}] a]
    set result2 [dict get [simulate $input [list b $result1]] a]
    yield $result1 
    yield $result2
}

Part1	16076 (354142 us)
Part2	2797 (4 us)
